In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel('online_retail.xlsx')

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [7]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [8]:
df = df.dropna()

In [9]:
df.shape

(406829, 8)

In [10]:
df = df[df['Quantity'] > 0]

In [11]:
df = df[df['UnitPrice'] > 0]

In [12]:
df.shape

(397884, 8)

In [13]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [14]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [15]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour


# **Demand per Product**

In [16]:
product_demand = df.groupby('StockCode')['Quantity'].sum().reset_index()
product_demand.columns = ['StockCode', 'TotalDemand']

In [17]:
df = df.merge(product_demand, on='StockCode', how='left')

In [18]:
df['PriceCategory'] = pd.cut(df['UnitPrice'],
                            bins=[0, 2, 5, 10, 50, 1000],
                            labels=['Very Low', 'Low', 'Medium', 'High', 'Premium'])

In [20]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Day,Hour,TotalDemand,PriceCategory
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,8,36782,Low
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,1639,Low
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,8,1918,Low
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,2469,Low
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,3707,Low


# **Average Demand per Price**

In [21]:
price_demand = df.groupby('UnitPrice')['Quantity'].mean().reset_index()
price_demand.columns = ['Price', 'AvgDemand']

price_demand.head()

,Price,AvgDemand
0,0.001,1.000000
1,0.040,130.303030
2,0.060,124.928571
3,0.070,468.571429
4,0.080,71.890909


In [22]:
df[['UnitPrice', 'Quantity']].corr()

,UnitPrice,Quantity
UnitPrice,1.000000,-0.004563
Quantity,-0.004563,1.000000


# **Revenue per Product**

In [23]:
revenue_product = df.groupby('StockCode')['TotalPrice'].sum().reset_index()
revenue_product.columns = ['StockCode', 'TotalRevenue']

revenue_product.head()

,StockCode,TotalRevenue
0,10002,699.55
1,10080,114.41
2,10120,40.53
3,10125,930.30
4,10133,1143.61


In [24]:
df = df.merge(revenue_product, on='StockCode', how='left')

In [25]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Day,Hour,TotalDemand,PriceCategory,TotalRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,8,36782,Low,100603.50
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,1639,Low,5915.95
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,8,1918,Low,7125.38
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,2469,Low,9329.17
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8,3707,Low,13430.51


In [26]:
features = ['UnitPrice', 'Month', 'Day', 'Hour', 'TotalDemand']
target = 'Quantity'

In [27]:
X = df[features]
y = df[target]

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [29]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [30]:
y_pred = model.predict(X_test)

In [31]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 7.94400882387358
R2 Score: 0.2912641557814446


In [32]:
df['DemandCategory'] = pd.qcut(df['TotalDemand'], q=3, labels=['Low', 'Medium', 'High'])

In [33]:
def recommend_price(row):
    if row['DemandCategory'] == 'High':
        return row['UnitPrice'] * 1.1   # increase 10%
    elif row['DemandCategory'] == 'Low':
        return row['UnitPrice'] * 0.9   # decrease 10%
    else:
        return row['UnitPrice']         # keep same

In [34]:
df['RecommendedPrice'] = df.apply(recommend_price, axis=1)

In [35]:
df['NewRevenue'] = df['RecommendedPrice'] * df['Quantity']

In [36]:
old_revenue = df['TotalPrice'].sum()
new_revenue = df['NewRevenue'].sum()

print("Old Revenue:", old_revenue)
print("New Revenue:", new_revenue)

Old Revenue: 8911407.904
New Revenue: 9058699.2346


In [37]:
df.to_csv('final_dynamic_pricing_data.csv', index=False)